# 04 — Candidate Generation: Collaborative Filtering (ALS) + Optuna Tuning

**Purpose:** the 'bread and butter' collaborative source, with ALS hyperparameters
chosen by Optuna instead of guessed.

- Removed the old item-subsampling and the Optuna-over-subsampling-fractions hack.
- Optuna now tunes the **real** ALS knobs — `factors`, `iterations`, `regularization`,
  and confidence `alpha` — by training on the VAL basis (`t_dat < val_week_start`) and
  maximizing **MAP@12** against `val_truth`. No leakage.
- The winning params are then used to generate candidates for **both** label weeks.

In [1]:
%pip install optuna

Note: you may need to restart the kernel to use updated packages.


In [2]:
from recsys_utils import *
import numpy as np, pandas as pd, pickle
from scipy.sparse import csr_matrix
from implicit.als import AlternatingLeastSquares
import optuna
mlflow = setup_mlflow()

transactions = load_parquet(PROCESSED_DIR / 'transactions_clean.parquet')
cfg          = load_json(PROCESSED_DIR / 'split_config.json')
train_truth  = load_parquet(PROCESSED_DIR / 'train_truth.parquet')
val_truth    = load_parquet(PROCESSED_DIR / 'val_truth.parquet')
# If optuna isn't installed yet:  %pip install optuna

In [3]:
ALS_TOPN   = 100    # candidates exported per user
N_TRIALS   = 20     # Optuna trials (lower to ~10 if CPU-bound; faster with use_gpu)
RUN_OPTUNA = True   # set False on later re-runs to reuse BEST_PARAMS below

# Fallback used only if RUN_OPTUNA = False
BEST_PARAMS = {'factors': 64, 'regularization': 0.01, 'iterations': 20, 'alpha': 15.0}

## ALS building blocks (shared by tuning and final generation)

The matrix uses **confidence weighting** (Hu et al.): cell value = `1 + alpha * purchase_count`,
so repeat purchases count for more and `alpha` becomes a meaningful knob to tune.

In [4]:
def build_ui(basis_end, alpha):
    h = transactions.loc[transactions['t_dat'] < basis_end, ['customer_id', 'article_id']]
    g = h.groupby(['customer_id', 'article_id']).size().reset_index(name='cnt')
    u_cat = pd.Categorical(g['customer_id'])
    i_cat = pd.Categorical(g['article_id'])
    data  = (1.0 + alpha * g['cnt'].to_numpy()).astype('float32')
    ui = csr_matrix((data, (u_cat.codes, i_cat.codes)),
                    shape=(len(u_cat.categories), len(i_cat.categories)))
    user_pos  = pd.Series(np.arange(len(u_cat.categories)), index=np.asarray(u_cat.categories))
    item_cats = np.asarray(i_cat.categories)
    return ui, user_pos, item_cats

def fit_als(ui, factors, regularization, iterations):
    m = AlternatingLeastSquares(factors=factors, regularization=regularization,
                                iterations=iterations, random_state=RANDOM_SEED)
    m.fit(ui, show_progress=False)   # add use_gpu=True in the constructor if you built implicit w/ CUDA
    return m

def recommend_df(model, ui, user_pos, item_cats, users, topn):
    su = np.intersect1d(np.asarray(users), user_pos.index.values)
    li = user_pos.loc[su].values
    ids, scores = model.recommend(li, ui[li], N=topn, filter_already_liked_items=True)
    n = len(su)
    out = pd.DataFrame({
        'customer_id': np.repeat(su, topn).astype('int32'),
        'article_id':  item_cats[ids.ravel()].astype('int32'),
        'als_score':   scores.ravel().astype('float32'),
        'als_rank':    np.tile(np.arange(1, topn + 1), n).astype('int16'),
    })
    out['source'] = 'als'
    return out

## Optuna study — maximize MAP@12 on the holdout week

In [5]:
val_basis = pd.Timestamp(cfg['val_week_start'])
val_users = val_truth['customer_id'].to_numpy()
truth_map = dict(zip(val_truth['customer_id'], val_truth['actual']))
val_user_list = val_truth['customer_id'].tolist()
val_actuals   = val_truth['actual'].tolist()

def evaluate_als(factors, regularization, iterations, alpha):
    ui, user_pos, item_cats = build_ui(val_basis, alpha)
    model = fit_als(ui, factors, regularization, iterations)
    recs  = recommend_df(model, ui, user_pos, item_cats, val_users, TOP_K)
    rec_map = recs.groupby('customer_id')['article_id'].apply(list).to_dict()
    preds = [rec_map.get(u, []) for u in val_user_list]
    return mapk(val_actuals, preds, TOP_K)

def objective(trial):
    factors = trial.suggest_int('factors', 32, 256, step=32)
    reg     = trial.suggest_float('regularization', 1e-3, 1e-1, log=True)
    iters   = trial.suggest_int('iterations', 10, 40)
    alpha   = trial.suggest_float('alpha', 1.0, 40.0, log=True)
    return evaluate_als(factors, reg, iters, alpha)

In [6]:
if RUN_OPTUNA:
    optuna.logging.set_verbosity(optuna.logging.WARNING)
    study = optuna.create_study(direction='maximize')
    study.optimize(objective, n_trials=N_TRIALS, show_progress_bar=True)
    BEST_PARAMS = study.best_params
    print('Best MAP@12 :', round(study.best_value, 5))
    print('Best params :', BEST_PARAMS)
else:
    print('Using preset BEST_PARAMS:', BEST_PARAMS)

  0%|          | 0/20 [00:00<?, ?it/s]

C:\Users\Michael Fulling\anaconda3\Lib\site-packages\implicit\cpu\als.py:96: RuntimeWarning: Intel MKL BLAS is configured to use 24 threads. It is highly recommended to disable its internal threadpool by setting the environment variable 'MKL_NUM_THREADS=1' or by callng 'threadpoolctl.threadpool_limits(1, "blas")'. Having MKL use a threadpool can lead to severe performance issues
  check_blas_config()


Best MAP@12 : 0.0043
Best params : {'factors': 128, 'regularization': 0.003546981655846009, 'iterations': 22, 'alpha': 16.21902250736541}


## Generate ALS candidates for both weeks using the tuned params

In [7]:
weeks = [
    ('train', train_truth['customer_id'].to_numpy(), pd.Timestamp(cfg['train_week_start'])),
    ('val',   val_truth['customer_id'].to_numpy(),   pd.Timestamp(cfg['val_week_start'])),
]

for tag, users, basis in weeks:
    ui, user_pos, item_cats = build_ui(basis, BEST_PARAMS['alpha'])
    model = fit_als(ui, BEST_PARAMS['factors'], BEST_PARAMS['regularization'], BEST_PARAMS['iterations'])
    cand  = recommend_df(model, ui, user_pos, item_cats, users, ALS_TOPN)
    save_parquet(cand, CANDIDATE_DIR / f'als_candidates_{tag}.parquet')
    print(f'[{tag}] ALS rows={len(cand):,}  users={cand.customer_id.nunique():,}  items={cand.article_id.nunique():,}')
    if tag == 'val':
        with open(MODEL_DIR / 'als_model_val.pkl', 'wb') as f:
            pickle.dump({'model': model, 'user_cats': np.asarray(user_pos.index.values),
                         'item_cats': item_cats, 'params': BEST_PARAMS}, f)
        print('Saved als_model_val.pkl')

Saved: C:\Users\Michael Fulling\H&M Project\data\processed\candidates\als_candidates_train.parquet  shape=(6662400, 5)
[train] ALS rows=6,662,400  users=66,624  items=13,364
Saved: C:\Users\Michael Fulling\H&M Project\data\processed\candidates\als_candidates_val.parquet  shape=(6341200, 5)
[val] ALS rows=6,341,200  users=63,412  items=13,402
Saved als_model_val.pkl


In [8]:
with mlflow.start_run(run_name='04_candidates_als_optuna'):
    mlflow.log_params({f'als_{k}': v for k, v in BEST_PARAMS.items()})
    mlflow.log_param('als_topn', ALS_TOPN)
    mlflow.log_param('n_trials', N_TRIALS if RUN_OPTUNA else 0)
    if RUN_OPTUNA:
        mlflow.log_metric('als_best_map_at_12', float(study.best_value))
print('Logged ALS+Optuna run.')

2026/06/11 12:57:24 WARNING mlflow.utils.git_utils: Failed to import Git (the Git executable is probably not on your PATH), so Git SHA is not available. Error: Failed to initialize: Bad git executable.
The git executable must be specified in one of the following ways:
    - be included in your $PATH
    - be set via $GIT_PYTHON_GIT_EXECUTABLE
    - explicitly set via git.refresh(<full-path-to-git-executable>)

All git commands will error until this is rectified.

This initial message can be silenced or aggravated in the future by setting the
$GIT_PYTHON_REFRESH environment variable. Use one of the following values:
    - quiet|q|silence|s|silent|none|n|0: for no message or exception
    - warn|w|warning|log|l|1: for a warning message (logging level CRITICAL, displayed by default)
    - error|e|exception|raise|r|2: for a raised exception

Example:
    export GIT_PYTHON_REFRESH=quiet



Logged ALS+Optuna run.
